# MLE for the Gaussian

Two observations, $1$ and $-1$, sampled from *some* normal distribution. Which one? For continuous data the likelihood of an observation is the **height of the candidate's PDF** at that point, and independent observations multiply their heights. Racing candidate Gaussians against each other reveals two beautiful facts: the winning **mean** is the *sample mean* of the data, and the winning **variance** is the *sample variance* (the divide-by-$n$ one!). It is based on the fourth video of Lesson 6 (Point Estimation).

## 1. Introduction

The coin notebook maximized likelihoods of discrete outcomes. Continuous data changes one thing only: probabilities of exact values are zero, so the likelihood of an observation under a candidate distribution is the **PDF evaluated at that observation** — the height of the curve above the point. Everything else (compare candidates, multiply across independent observations, pick the argmax) carries over unchanged.

After completing this notebook you will be able to:

- Read likelihoods off a Gaussian curve as heights, and multiply them across independent observations.
- Run the video's two contests: candidate **means** (winner: the sample mean) and candidate **standard deviations** (winner: the sample variance).
- Fit both parameters at once on a likelihood surface, and state the general Gaussian MLE: $\hat\mu = \bar{x}$, $\hat\sigma^2 = \frac1n\sum(x_i - \bar x)^2$.
- Connect the $\div n$ in the MLE variance back to the sample-variance notebook's bias discussion.

## 2. Theory

### Heights are likelihoods

Observations: $\{1, -1\}$. First contest: $N(10, 1)$ versus $N(2, 1)$. Plot both curves and look at their heights above the two points. Under $N(10, 1)$ the heights are astronomically small (the points sit 9 and 11 standard deviations away); under $N(2, 1)$ they are much larger. These heights are — well, not exactly probabilities — **likelihoods**, and the curve that produces both points with higher likelihood wins: $N(2, 1)$.

### Contest of means

Now three candidates, all with $\sigma = 1$: means $-1$, $0$, and $1$. The likelihood bars (point up to curve):

| candidate | height at $-1$ | height at $1$ | product |
|---|---|---|---|
| $N(-1, 1)$ | 0.399 | 0.054 | 0.022 |
| $N(0, 1)$ | 0.242 | 0.242 | **0.059** |
| $N(1, 1)$ | 0.054 | 0.399 | 0.022 |

Why multiply? Because the two points were **independently** generated — for probabilities we would multiply, so for likelihoods we multiply too. The winner is $N(0, 1)$.

And notice: the winning mean $0$ **is the sample mean of the data** $\frac{1 + (-1)}{2} = 0$. The best distribution is the one whose mean is the mean of the sample.

### Contest of standard deviations

Fix the mean at the established $0$; race $\sigma = 0.5$, $1$, $2$:

| candidate | height at $\pm1$ | product |
|---|---|---|
| $N(0, 0.5^2)$ | 0.108 | 0.012 |
| $N(0, 1)$ | 0.242 | **0.059** |
| $N(0, 2^2)$ | 0.176 | 0.031 |

$\sigma = 0.5$ hugs the origin so tightly that the points sit in its thin tails; $\sigma = 2$ spreads its height over too wide a range. The winner is again $\sigma = 1$ — and notice: the **variance of the observations** (average squared distance from their mean: $\frac{1^2 + (-1)^2}{2} = 1$) matches the winning variance.

### The general result

For data $x_1, \dots, x_n$ and Gaussian candidates $N(\mu, \sigma^2)$, maximizing the (log-)likelihood gives

$$
\hat{\mu} = \bar{x} = \frac1n \sum_i x_i
\qquad\qquad
\hat{\sigma}^2 = \frac{1}{n}\sum_i (x_i - \bar{x})^2
$$

— the sample mean, and the **divide-by-$n$** sample variance. (Recall the sample-variance notebook: the unbiased estimator divides by $n-1$, but "some accepted techniques use $n$ — for example maximum likelihood estimation." Here is that promise kept: the MLE genuinely prefers $\div n$.) The course's follow-up reading derives this with calculus; the exercises below reproduce it with sympy.

## 3. Worked Examples

### Example 1: the lopsided first contest

Under $N(10,1)$, the height at $x = 1$ is $\varphi(9) \approx 10^{-18}$ — under $N(2,1)$ it is $\varphi(1) \approx 0.242$. One glance at the curves settles it; likelihood just quantifies the glance.

### Example 2: the symmetry of the means contest

$N(-1,1)$ and $N(1,1)$ tie exactly (0.022 each): each nails one point (height 0.399) and badly misses the other (0.054). $N(0,1)$ compromises at 0.242 each — and the *product* rewards balance: $0.242^2 = 0.059 > 0.399 \cdot 0.054 = 0.022$. Maximizing a product favors explaining *all* the data adequately over explaining some of it perfectly.

### Example 3: the variance of $\{1, -1\}$

Sample mean $0$; squared deviations $1$ and $1$; average $= 1$. So the MLE Gaussian is $N(0, 1)$ — both contests were won by the data's own moments. (With $n = 2$, the $\div(n{-}1)$ estimate would be $2$ instead — visibly different at tiny $n$, exactly as the practitioner's rule warned.)

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

def normal_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))

data = np.array([1.0, -1.0])

### The three contests, computed

In [ ]:
print("contest 0: N(10,1) vs N(2,1)")
for mu in [10, 2]:
    h = normal_pdf(data, mu, 1)
    print(f"  N({mu:>2},1): heights {h.round(6)}  product {h.prod():.2e}")

print("\ncontest 1: three means (sigma = 1)")
for mu in [-1, 0, 1]:
    h = normal_pdf(data, mu, 1)
    print(f"  N({mu:>2},1): heights {h.round(3)}  product {h.prod():.3f}")

print("\ncontest 2: three sigmas (mean = 0)")
for s in [0.5, 1, 2]:
    h = normal_pdf(data, 0, s)
    print(f"  N(0,{s}): heights {h.round(3)}  product {h.prod():.3f}")

print("\nWinners: mean 0 (the sample mean) and sigma 1 (the sample variance).")

### The two winning numbers are the data's own moments

In [ ]:
mu_hat = data.mean()
sigma2_hat = np.mean((data - mu_hat)**2)          # MLE: divide by n
print(f"sample mean of {data}: {mu_hat}")
print(f"MLE variance (/n):     {sigma2_hat}")
print(f"unbiased s^2 (/(n-1)): {np.var(data, ddof=1)}   <- NOT what MLE picks")

### Fitting both parameters at once: the likelihood surface

Grid over $(\mu, \sigma)$ and evaluate the product of heights for the two points — the peak sits at $(0, 1)$, both contests solved simultaneously.

In [ ]:
mus = np.linspace(-2, 2, 401)
sigmas = np.linspace(0.3, 3, 271)
M, S = np.meshgrid(mus, sigmas)

log_lik = sum(-0.5 * ((x - M) / S)**2 - np.log(S * np.sqrt(2 * np.pi))
              for x in data)
i, j = np.unravel_index(np.argmax(log_lik), log_lik.shape)
print(f"grid argmax: mu = {M[i, j]:.2f}, sigma = {S[i, j]:.2f}")
print(f"theory: mu_hat = {data.mean():.2f}, "
      f"sigma_hat = {np.sqrt(np.mean((data - data.mean())**2)):.2f}")

### The general formulas, verified on a big sample

In [ ]:
rng = np.random.default_rng(0)

big = rng.normal(7.3, 2.1, size=100_000)

mus = np.linspace(6.8, 7.8, 201)
sigmas = np.linspace(1.6, 2.6, 201)
# maximize over mu at the true-ish sigma and vice versa is fine at this n;
# use the closed forms instead and check by local perturbation
mu_hat = big.mean()
sigma_hat = np.sqrt(np.mean((big - mu_hat)**2))

def loglik(mu, s):
    return np.sum(-0.5 * ((big - mu) / s)**2 - np.log(s))

print(f"closed-form MLE: mu = {mu_hat:.4f}, sigma = {sigma_hat:.4f}")
print(f"log-lik at the MLE:        {loglik(mu_hat, sigma_hat):,.1f}")
print(f"log-lik nudging mu +0.01:  {loglik(mu_hat + 0.01, sigma_hat):,.1f}")
print(f"log-lik nudging sigma +0.01: {loglik(mu_hat, sigma_hat + 0.01):,.1f}")
print("Any nudge lowers the log-likelihood: (x_bar, /n variance) is the peak.")

## 5. Visualizations

### Contest 0 and contest 1: heights as vertical bars

The video's pictures — candidate curves with the likelihood bars rising from each observation.

In [ ]:
x_grid = np.linspace(-4.5, 4.5, 500)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, mu in zip(axes, [-1, 0, 1]):
    ax.plot(x_grid, normal_pdf(x_grid, mu, 1), color="tab:blue", linewidth=2)
    prod = 1.0
    for x in data:
        h = normal_pdf(x, mu, 1)
        prod *= h
        ax.plot([x, x], [0, h], color="tab:red", linewidth=3)
        ax.plot(x, 0, "o", color="black", markersize=8)
        ax.annotate(f"{h:.3f}", xy=(x, h), xytext=(x + 0.15, h + 0.015), fontsize=9)
    ax.set_title(f"N({mu}, 1): product = {prod:.3f}")
    ax.set_xlabel("x")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("density")
plt.suptitle("Contest of means for the data {-1, 1}: the sample-mean Gaussian wins")
plt.tight_layout()
plt.show()

### Contest 2: too narrow, just right, too wide

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, s in zip(axes, [0.5, 1, 2]):
    ax.plot(x_grid, normal_pdf(x_grid, 0, s), color="tab:green", linewidth=2)
    prod = 1.0
    for x in data:
        h = normal_pdf(x, 0, s)
        prod *= h
        ax.plot([x, x], [0, h], color="tab:red", linewidth=3)
        ax.plot(x, 0, "o", color="black", markersize=8)
        ax.annotate(f"{h:.3f}", xy=(x, h), xytext=(x + 0.15, h + 0.015), fontsize=9)
    ax.set_title(f"N(0, {s}$^2$): product = {prod:.3f}")
    ax.set_xlabel("x")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("density")
plt.suptitle("Contest of spreads: sigma = 0.5 buries the points in its tails, sigma = 2 dilutes -- sigma = 1 wins")
plt.tight_layout()
plt.show()

### The full likelihood surface over $(\mu, \sigma)$

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
levels = np.linspace(log_lik.max() - 8, log_lik.max(), 25)
cs = ax.contourf(M, S, log_lik, levels=levels, cmap="viridis")
ax.plot(0, 1, "r*", markersize=20, label="MLE: $(\\hat\\mu, \\hat\\sigma) = (0, 1)$")
for mu in [-1, 0, 1]:
    ax.plot(mu, 1, "wo", markersize=6)
for s in [0.5, 2]:
    ax.plot(0, s, "ws", markersize=6)
ax.set_xlabel("$\\mu$")
ax.set_ylabel("$\\sigma$")
ax.set_title("Log-likelihood of {-1, 1}: the contests were 1-D slices of this surface")
fig.colorbar(cs, label="log-likelihood")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** For observations $\{3, 7\}$, race the candidates $N(4, 1)$, $N(5, 1)$, $N(6, 1)$ by likelihood products, and confirm the winner's mean is the sample mean. What $\hat\sigma$ would the full MLE report?

**Exercise 2.** Explain in one sentence each: (a) why heights instead of probabilities for continuous data; (b) why the heights get *multiplied* across observations.

### Intermediate

**Exercise 3.** Derive the Gaussian MLE with sympy: write the log-likelihood for $n$ observations, differentiate with respect to $\mu$ and $\sigma$, and solve — confirming $\hat\mu = \bar x$ and $\hat\sigma^2 = \frac1n\sum(x_i-\bar x)^2$. (Two symbolic observations $a, b$ are enough to see the pattern.)

**Exercise 4.** The $\div n$ vs $\div(n-1)$ reunion: simulate many size-5 samples from $N(0, 1)$, compute the MLE variance and $s^2$ for each, and compare their averages with the true $\sigma^2 = 1$. Which estimator is biased, in which direction, and by what factor?

### Advanced

**Exercise 5.** One observation, degenerate MLE: for the single data point $\{4\}$, what do $\hat\mu$ and $\hat\sigma^2$ come out to? Explore the likelihood as $\sigma \to 0$ and explain why the MLE "blows up" — the continuous cousin of the coin's 10-of-10 overfit.

**Exercise 6.** MLE with the *wrong* model family: generate 10,000 points from a uniform$(0, 10)$ and fit a Gaussian by MLE. What $(\hat\mu, \hat\sigma)$ do you get, and how do they relate to the uniform's own mean and variance? What does this say about what MLE does when the truth isn't in the candidate set?

## 7. Solutions

In [ ]:
# Exercise 1: the {3, 7} contest
d2 = np.array([3.0, 7.0])
for mu in [4, 5, 6]:
    prod = normal_pdf(d2, mu, 1).prod()
    print(f"N({mu},1): product = {prod:.6f}")
print(f"winner mean = 5 = sample mean {d2.mean()}")
print(f"full MLE sigma = sqrt(mean squared deviation) = "
      f"{np.sqrt(np.mean((d2 - d2.mean())**2))}")

In [ ]:
# Exercise 2: two one-liners
print("Exercise 2:")
print("(a) Exact continuous values have probability zero, so we compare the")
print("    DENSITY heights, which play the role of likelihood for observations.")
print("(b) The observations are independent, and joint likelihood of independent")
print("    data is the product of individual likelihoods -- same as probabilities.")

In [ ]:
# Exercise 3: symbolic derivation
mu_s = sp.Symbol("mu")
sig_s = sp.symbols("sigma", positive=True)
a_s, b_s = sp.symbols("a b")
logL = sum(-sp.log(sig_s) - sp.Rational(1, 2) * sp.log(2 * sp.pi)
           - (x - mu_s)**2 / (2 * sig_s**2) for x in (a_s, b_s))
mu_sol = sp.solve(sp.Eq(sp.diff(logL, mu_s), 0), mu_s)[0]
print(f"Exercise 3: mu_hat = {mu_sol}   (= (a+b)/2 = the sample mean)")

# solve for the variance v = sigma^2 directly (cleaner than solving for sigma)
v = sp.symbols("v", positive=True)
logL_v = sum(-sp.Rational(1, 2) * sp.log(v) - (x - mu_sol)**2 / (2 * v)
             for x in (a_s, b_s))
v_sol = sp.simplify(sp.solve(sp.Eq(sp.diff(logL_v, v), 0), v)[0])
print(f"sigma_hat^2 = {v_sol}")
print("= ((a-b)/2)^2 = the /n mean squared deviation of two points.")

In [ ]:
# Exercise 4: the bias factor, again
rng = np.random.default_rng(1)

samples = rng.normal(0, 1, size=(100_000, 5))
mle_var = samples.var(axis=1, ddof=0)
s2 = samples.var(axis=1, ddof=1)
print(f"Exercise 4: average MLE variance (/n):   {mle_var.mean():.4f}")
print(f"            average s^2 (/(n-1)):        {s2.mean():.4f}")
print(f"MLE is biased LOW by the familiar factor (n-1)/n = {4/5} -- maximum")
print("likelihood knowingly trades a little bias for being the likelihood peak.")

In [ ]:
# Exercise 5: one point, degenerate fit
point = 4.0
sigmas = np.array([1.0, 0.1, 0.01, 0.001])
heights = normal_pdf(point, point, sigmas)
print("Exercise 5: mu_hat = 4 (the point itself); sigma_hat^2 = 0!")
for s, h in zip(sigmas, heights):
    print(f"  likelihood at sigma = {s}: {h:,.1f}")
print("The density height at the point grows without bound as sigma -> 0: the")
print("'best' Gaussian is an infinitely tall spike on the single observation --")
print("perfect fit, zero generalization. Small-n MLE overfits; priors (MAP) or")
print("more data are the cures.")

In [ ]:
# Exercise 6: fitting a Gaussian to uniform data
rng = np.random.default_rng(2)

u = rng.uniform(0, 10, 10_000)
mu_hat_u = u.mean()
sigma_hat_u = np.sqrt(np.mean((u - mu_hat_u)**2))
print(f"Exercise 6: Gaussian MLE on uniform data: mu = {mu_hat_u:.3f}, "
      f"sigma = {sigma_hat_u:.3f}")
print(f"uniform's own mean = 5, std = sqrt(100/12) = {np.sqrt(100/12):.3f}")
print("MLE picks the Gaussian matching the data's mean and variance -- the")
print("closest member of the (wrong) family. MLE always answers 'best available")
print("candidate', never 'is this family right?' -- checking fit is a separate")
print("job (QQ plots!).")

## 8. Conclusion

- For continuous data, likelihoods are **PDF heights** at the observations, and independent observations **multiply** their heights — after that, MLE proceeds exactly as with coins.
- The contests for $\{1, -1\}$: among means, $N(0,1)$ wins ($0.242^2 = 0.059$ beats $0.399 \cdot 0.054 = 0.022$) — and $0$ is the **sample mean**; among spreads, $\sigma = 1$ wins (0.059 vs 0.012 and 0.031) — and $1$ is the **sample variance** $\frac{1^2 + (-1)^2}{2}$.
- General Gaussian MLE: $\hat\mu = \bar x$ and $\hat\sigma^2 = \frac1n\sum(x_i - \bar x)^2$ — the promised technique that legitimately divides by $n$ (biased low by $\frac{n-1}{n}$, but the honest likelihood maximizer).
- The product-of-heights criterion rewards *balanced* explanation of all data over perfect explanation of some — and warns of degeneracies at tiny $n$ (a single point drives $\hat\sigma \to 0$).
- MLE finds the best candidate *within the family you offer*; whether the family fits at all is a different question (QQ plots, Lesson 3).

Next: MLE with the model family that runs machine learning — **linear regression**, where maximizing Gaussian likelihood becomes minimizing squared errors.